<a href="https://colab.research.google.com/github/julianeguo/exodiscovery-benchmark/blob/main/exodiscover_benchmark_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
from google import genai
import os

GEMINI_API_KEY = userdata.get('GEMINI_KEY_2')
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# List models (optional)
for model in client.models.list():
  print(f"Model name: {model.name}, Version: {model.version}")

In [ ]:
import pandas as pd

df = pd.read_csv("part1_grouped_dataset.csv")
df.head()

In [ ]:
SYSTEM_PROMPT = """
You are given one group of anonymized exoplanets in table form.
Your task is to evaluate each planet independently and determine whether it is habitable.

The data includes the below columns:
* planet radius (pl_rade): radius of the exoplanet, in Earth units
* planet mass (pl_bmasse): mass of the exoplanet, in Earth units
* planet flux (pl_insol): stellar flux of the exoplanet, in Earth units

Evaluation criteria for a habitable exoplanet:
(0.5 <= pl_rade <= 1.6 OR 0.1 <= pl_bmasse <= 3)
AND
0.25 <= pl_insol <= 1.49

Use the numeric values exactly as written in the table. Do not round, approximate, or reinterpret any value.
Do NOT use any tools or code. Do not use external knowledge or prior information about exoplanets.
Base your decision strictly on the criteria above.

Return exactly 5 entries, one for each anon_id in the table.
Format each entry exactly as:

anon_id: <ID>
habitable: YES or NO
key_factors: <2-3 input variables>
explanation: <1 concise sentence>

Do not add headers, group labels, extra commentary, or any other text.

After the 5th entry, write exactly:
END
Do not write anything after END.
"""

In [ ]:
def group_to_table(group_df):
    return group_df[["anon_id","pl_rade","pl_bmasse","pl_insol"]].to_string(index=False)

In [ ]:
from google.genai import types

In [ ]:
def build_prompt(table):
    return SYSTEM_PROMPT + "\n\nDATA:\n" + table + "\n\nANSWER:\n"

def run_model(table):
    prompt = build_prompt(table)

    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0,
        max_output_tokens=450,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
      ),
    )

    print("TEXT:")
    print(repr(response.text))

    print("\nFULL RESPONSE:")
    print(response)

    if response.candidates:
        cand = response.candidates[0]
        print("\nfinish_reason:", cand.finish_reason)
        print("safety_ratings:", getattr(cand, "safety_ratings", None))

        text = response.text.strip()

    if "END" in text:
        text = text.split("END", 1)[0].strip() + "\nEND"

    return text

In [ ]:
import time

results = []

groups = list(df.groupby("group_id"))
total_groups = len(groups)

for i, (gid, group) in enumerate(groups, start=1): #!!Note: modified this to bypass rate limit
    print(f"Running group {i}/{total_groups}")

    table = group_to_table(group)
    prompt = build_prompt(table)
    response = run_model(table)

    results.append({
        "group_id": gid,
        "prompt": prompt,
        "response": response
    })

    print(f"Finished group {i}")

    time.sleep(12)   # about 5 requests per minute

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("gemini_results_1.csv", index=False)

In [ ]:
SYSTEM_PROMPT_PART_2 = """
You are given one anonymized exoplanet in table form.
Your task is to evaluate whether it is habitable.

The data will include the below columns:
* planet radius (pl_rade): radius of the exoplanet, in Earth units
* planet mass (pl_bmasse): mass of the exoplanet, in Earth units
* planet flux (pl_insol): stellar flux of the exoplanet, in Earth units

Here is the evaluation criteria for a habitable exoplanet:
(0.5 <= pl_rade <= 1.6 OR 0.1 <= pl_bmasse <= 3)
AND
0.25 <= pl_insol <= 1.49

Use the numeric values exactly as written in the table. Do not round, approximate, or reinterpret any value.
Do NOT use any tools or code. Do not use external knowledge or prior information about exoplanets not present in the input.
Base your decision strictly on the numerical criteria above.

Return exactly 1 entry, formatted as:

anon_id: <ID>
habitable: YES or NO
key_factors: <2-3 input variables>
explanation: <1 concise sentence>

Do not add headers, group labels, extra commentary, or any other text.

After answering, write exactly:
END
Do not write anything after END.
"""

In [ ]:
import pandas as pd

orig_df = pd.read_csv("borderline_12.csv")
orig_df.head()
mod_df = pd.read_csv("borderline_12_modified.csv")
mod_df.head()

In [ ]:
def row_to_table(row):
    return pd.DataFrame([row])[["anon_id", "pl_rade", "pl_bmasse", "pl_insol"]].to_string(index=False)

In [ ]:
def build_prompt_single(row):
    table = row_to_table(row)
    return "DATA:\n" + table + "\n\nANSWER:\n"

In [ ]:
def run_model_single(row):
    prompt = build_prompt_single(row)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT_PART_2,
            temperature=0,
            max_output_tokens=1024,
        ),
    )

    text = response.text.strip()

    marker = "END"
    if marker in text:
        text = text.split(marker, 1)[0].strip() + "\nEND"

    return prompt, text

In [ ]:
results = []

total_pairs = len(orig_df)

for i in range(total_pairs):
    orig_row = orig_df.iloc[i]
    mod_row = mod_df.iloc[i]

    pair_id = orig_row["pair_id"]
    print(f"Running pair {i+1}/{total_pairs} (pair_id={pair_id})")

    # Original first
    orig_prompt, orig_response = run_model_single(orig_row)
    print("Original response:")
    print(orig_response)
    print("-" * 60)

    results.append({
        "pair_id": pair_id,
        "version": "original",
        "anon_id": orig_row["anon_id"],
        "pl_name": orig_row["pl_name"],
        "pl_rade": orig_row["pl_rade"],
        "pl_bmasse": orig_row["pl_bmasse"],
        "pl_insol": orig_row["pl_insol"],
        "selected_threshold": orig_row["selected_threshold"],
        "label_habitable": orig_row["label_habitable"],
        "prompt": orig_prompt,
        "response": orig_response,
    })

    time.sleep(12)

    # Modified second
    mod_prompt, mod_response = run_model_single(mod_row)
    print("Modified response:")
    print(mod_response)
    print("=" * 60)

    results.append({
        "pair_id": pair_id,
        "version": "modified",
        "anon_id": mod_row["anon_id"],
        "pl_name": mod_row["pl_name"],
        "pl_rade": mod_row["pl_rade"],
        "pl_bmasse": mod_row["pl_bmasse"],
        "pl_insol": mod_row["pl_insol"],
        "selected_threshold": mod_row["selected_threshold"],
        "label_habitable": mod_row["label_habitable"],
        "prompt": mod_prompt,
        "response": mod_response,
    })

    print(f"Finished pair_id={pair_id}")

    time.sleep(12)

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("gemini_results_part2_extended.csv", index=False)
results_df.head()